# Stage 1 — LIME Feature Extraction

**Reads:** Input texts defined in this notebook  
**Writes:** `outputs/lime_results.json`

Each record saved:
```json
{
  "text": "...",
  "fused_text": "...",
  "lime_features": [["hypertension", 0.42], ...]
}
```

> **Tip:** Run cells top to bottom on first run.  
> If `outputs/lime_results.json` already exists the checkpoint cell will warn you — set `FORCE_RERUN = True` to overwrite it.

## 1. Imports

In [1]:
from lime.lime_text import LimeTextExplainer

from config import (
    CLASS_NAMES,
    LIME_NUM_FEATURES,
    LIME_NUM_SAMPLES,
    LIME_RESULTS_PATH,
)
from model_loaders import load_classifier, load_ner_pipeline
from pipeline_helpers import (
    checkpoint_exists,
    load_checkpoint,
    make_lime_predictor,
    merge_entities,
    save_checkpoint,
)

## 2. Configuration

In [2]:
# Set True to re-run even if lime_results.json already exists
FORCE_RERUN = True

# ── Add / edit your input texts here ────────────────────────────────────────
with open("test_data.txt", "r", encoding="utf-8") as f:
    full_text = [line.strip() for line in f if line.strip()]

# First 200 texts
TEXTS = full_text[:50]


In [3]:
print(TEXTS)

['Normalization of ventilation/perfusion relationships after liver transplantation in patients with decompensated cirrhosis: evidence for a hepatopulmonary syndrome. To examine the effect of liver transplantation on the respiratory and cardiovascular functions, ventilation/perfusion relationships were determined by multiple inert gas elimination technique in six patients with end-stage liver disease 1 to 19 mo before and 2 to 6 mo after liver transplantation. Cardiac output and pulmonary vascular pressures were measured after catheterization of the pulmonary artery. All patients had normal spirometry and chest x-ray films before transplantation. PaO2 before transplantation was 78.8 +/- 7.4 mm Hg (range = 51.8 to 102.8 mm Hg). All patients had perfusion of poorly ventilated lung regions (low ventilation/perfusion relationships) varying from 3% to 19% of cardiac output (mean = 8.5% +/- 2.4% of cardiac output) and two patients had intrapulmonary shunting (3% and 20% of cardiac output). Me

## 3. Checkpoint check

In [4]:
if checkpoint_exists(LIME_RESULTS_PATH) and not FORCE_RERUN:
    print(f"⚠️  Checkpoint found at '{LIME_RESULTS_PATH}'.")
    print("    Set FORCE_RERUN = True in the cell above to overwrite.")
    print("    Loading existing results …")
    results = load_checkpoint(LIME_RESULTS_PATH)
else:
    results = None
    print("No checkpoint found (or FORCE_RERUN=True). Will run LIME.")

No checkpoint found (or FORCE_RERUN=True). Will run LIME.


## 4. Load models

In [5]:
if results is None:
    classifier_model, classifier_pipeline = load_classifier()
    ner_pipeline = load_ner_pipeline()

[Loader] Loading classifier from 'C:/Users/vimal/OneDrive/Documents/Uni/BTP/User-Adaptive-XAI/Models/my_medical_model' …


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[Loader] Classifier ready.

[Loader] Loading NER model 'd4data/biomedical-ner-all' …


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[Loader] NER model ready.



## 5. Run LIME

In [6]:
if results is None:
    explainer = LimeTextExplainer(class_names=CLASS_NAMES)
    predictor = make_lime_predictor(classifier_model, classifier_pipeline)

    results = []
    for i, text in enumerate(TEXTS):
        print(f"\nProcessing text {i + 1}/{len(TEXTS)} …")

        # Fuse multi-word biomedical entities before LIME
        fused_text = merge_entities(text, ner_pipeline)
        print(f"  Fused text preview: {fused_text[:120]}…")

        exp = explainer.explain_instance(
            fused_text,
            predictor,
            num_features=LIME_NUM_FEATURES,
            num_samples=LIME_NUM_SAMPLES,
        )

        # Restore underscores → spaces for downstream readability
        lime_features = [
            [w.replace("_", " "), float(score)]
            for w, score in exp.as_list()
        ]

        print(f"  Top features: {[f[0] for f in lime_features]}")

        results.append({
            "text":          text,
            "fused_text":    fused_text,
            "lime_features": lime_features,
        })

    print("\n✅ LIME complete.")


Processing text 1/50 …
  Fused text preview: Normalization of ventilation/perfusion_relationships after liver transplantation in patients with decompensated_cirrhosi…
  Top features: ['decompensated cirrhosis', 'stage liver disease', 'transplantation', '7', 'from', 'artery']

Processing text 2/50 …
  Fused text preview: Color doppler_imaging. A new noninvasive technique to diagnose and monitor carotid cavernous_sinus fistulas. Color doppl…
  Top features: ['arteriovenous malformations', 'doppler imaging', 'traumatic', 'cavernous sinus', 'development', 'fistulas']

Processing text 3/50 …
  Fused text preview: Endometriosis associated with massive ascites and absence of pelvic peritoneum. Although massive ascites associated with…
  Top features: ['ascites', 'peritoneum', 'abdominal', 'with', 'Endometriosis', 'massive']

Processing text 4/50 …
  Fused text preview: Long-term evaluation of indobufen in peripheral vascular disease. Indobufen--an inhibitor of platelets aggregation--has …
  

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  Top features: ['biliary', 'transhepatic', 'obstruction', 'bifurcational', 'with', 'The']

Processing text 11/50 …
  Fused text preview: lower_extremity arterial_disease in elderly subjects with systolic hypertension. The ratio of ankle-to-arm systolic bloo…
  Top features: ['systolic', 'aged 60 and older', 'greater than 160 mmhg', 'hypertension', 'appears', 'and']

Processing text 12/50 …
  Fused text preview: Role of surgery in antibiotic-induced pseudomembranous enterocolitis. With the increased use of prophylactic and broad-s…
  Top features: ['colitis', 'difficile', 'perforation', 'peritonitis', 'progress', 'iatrogenic']

Processing text 13/50 …
  Fused text preview: Patient age and results of balloon aortic valvuloplasty: the Mansfield Scientific Registry experience. The Mansfield Sci…
  Top features: ['heart failure', 'aortic', 'complications', 'Aortic', 'valvuloplasty', 'Valvuloplasty']

Processing text 14/50 …
  Fused text preview: How ancient is temporal arteritis? Realism i

## 6. Inspect results

In [7]:
for i, r in enumerate(results):
    print(f"\n── Text {i + 1} ──────────────────────────")
    print(f"Text preview : {r['text'][:100]}…")
    print(f"Top features : {r['lime_features']}")


── Text 1 ──────────────────────────
Text preview : Normalization of ventilation/perfusion relationships after liver transplantation in patients with de…
Top features : [['decompensated cirrhosis', 0.16537773138324022], ['stage liver disease', 0.0882714935041679], ['transplantation', -0.05239975695450586], ['7', 0.041646219198311886], ['from', -0.03319719409047882], ['artery', -0.03120806167272191]]

── Text 2 ──────────────────────────
Text preview : Color Doppler imaging. A new noninvasive technique to diagnose and monitor carotid cavernous sinus f…
Top features : [['arteriovenous malformations', -0.0016759673358058061], ['doppler imaging', -0.0014831974478600755], ['traumatic', -0.0012938818631146503], ['cavernous sinus', -0.0011829489372432451], ['development', -0.0009355713853331219], ['fistulas', 0.0008036988352175928]]

── Text 3 ──────────────────────────
Text preview : Endometriosis associated with massive ascites and absence of pelvic peritoneum. Although massive asc…
Top fe

## 7. Save checkpoint

In [8]:
save_checkpoint(results, LIME_RESULTS_PATH)
print(f"\n➡️  Continue to notebook 02_ontology.ipynb")

[Checkpoint] Saved 50 records → 'outputs\lr_exp.json'

➡️  Continue to notebook 02_ontology.ipynb
